In [5]:
# ── CELDA 1: Dependencias ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


In [ ]:
# ── CELDA 2: Carga y preparación de datos ────────────────────────────────────
from pathlib import Path
# Buscamos recursivamente archivos de datos en el workspace para mayor robustez
ROOT = Path.cwd().resolve()
SEARCH_NAMES = [
    'dataset_final_oro.parquet',
    'dataset_ia_clima_demanda.csv',
    'demanda_electrica_2013_2026.csv',
]
data_path = None
for name in SEARCH_NAMES:
    matches = list(ROOT.rglob(name))
    if matches:
        data_path = matches[0]
        break
if data_path is None:
    raise FileNotFoundError(f"No se encontró el fichero de datos. Buscados: {SEARCH_NAMES} dentro de {ROOT}")
# Cargar según extensión encontrada (soporta parquet y csv)
if data_path.suffix == '.parquet':
    df = pd.read_parquet(data_path)
else:
    try:
        df = pd.read_csv(data_path, parse_dates=['fecha'])
    except Exception:
        df = pd.read_csv(data_path)
        if 'fecha' in df.columns:
            df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
# Asegurar tipo fecha y ordenar
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# Garantizar columnas esperadas
if 'temp_med' not in df.columns:
    df['temp_med'] = (df['temp_max'] + df['temp_min']) / 2
if 'precipitacion' not in df.columns:
    df['precipitacion'] = 0.0
if 'es_festivo' not in df.columns:
    df['es_festivo'] = 0

# Columnas derivadas para análisis
df['anio']       = df['fecha'].dt.year
df['mes']        = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek   # 0=Lun … 6=Dom
df['nombre_mes'] = df['fecha'].dt.strftime('%b')
df['tipo_dia']   = df['es_festivo'].map({1: 'Festivo', 0: 'Laboral'})

# Paleta de colores del proyecto
COL_DEMANDA  = '#1B6CA8'
COL_FESTIVO  = '#E63946'
COL_TEMP     = '#F4A261'
COL_LLUVIA   = '#90C2E7'
COL_LABORAL  = '#2A9D8F'
BG_COLOR     = '#F8FAFC'
PAPER_COLOR  = '#FFFFFF'
FONT_COLOR   = '#1A1A2E'

AÑOS = sorted(df['anio'].unique().tolist())

print(f'✅ Dataset cargado: {len(df):,} registros | {df["anio"].min()} – {df["anio"].max()}')
print(f'   Columnas: {list(df.columns)}')

FileNotFoundError: No se encontró el fichero de datos. Buscados: ['/home/jovyan/work/dataset_final_oro.parquet', '/home/jovyan/work/dataset_ia_clima_demanda.csv', '/home/jovyan/work/demanda_electrica_2013_2026.csv']

In [ ]:
# ── CELDA 3: Estilo compartido ───────────────────────────────────────────────
LAYOUT_BASE = dict(
    font=dict(family='Source Sans Pro, Helvetica Neue, sans-serif', color=FONT_COLOR, size=13),
    paper_bgcolor=PAPER_COLOR,
    plot_bgcolor=BG_COLOR,
    hovermode='x unified',
    legend=dict(
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='#E0E0E0',
        borderwidth=1,
        font=dict(size=12)
    ),
    margin=dict(l=60, r=30, t=80, b=50),
)

AXIS_STYLE = dict(
    gridcolor='#E8EDF3',
    linecolor='#CBD5E0',
    tickfont=dict(size=11),
    showgrid=True,
    zeroline=False,
)

def titulo(texto, subtexto=''):
    """Genera título HTML con subtítulo opcional"""
    sub = f'<br><span style="font-size:13px;color:#718096">{subtexto}</span>' if subtexto else ''
    return f'<b>{texto}</b>{sub}'

print('✅ Estilos definidos')

---
## 📌 Panel 1 · Serie Temporal con Filtro por Año

In [ ]:
# ── CELDA 4: Panel 1 – Serie temporal interactiva ────────────────────────────

# ---- Widgets ----
w_anio_inicio = widgets.Dropdown(
    options=AÑOS, value=AÑOS[0],
    description='Desde:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='160px')
)
w_anio_fin = widgets.Dropdown(
    options=AÑOS, value=AÑOS[-1],
    description='Hasta:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='160px')
)
w_media_movil = widgets.IntSlider(
    value=30, min=7, max=90, step=7,
    description='Media móvil (días):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='380px')
)
w_mostrar_festivos = widgets.Checkbox(
    value=True, description='Marcar festivos',
    style={'description_width': 'initial'}
)
out1 = widgets.Output()

def render_panel1(anio_ini, anio_fin, ventana, mostrar_festivos):
    if anio_ini > anio_fin:
        with out1:
            out1.clear_output()
            print('⚠️  El año de inicio no puede ser mayor que el año de fin.')
        return

    d = df[(df['anio'] >= anio_ini) & (df['anio'] <= anio_fin)].copy()
    d['media_movil'] = d['demanda_mw'].rolling(ventana, center=True).mean()
    festivos = d[d['es_festivo'] == 1]

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=(
            'Consumo Eléctrico (MW)',
            'Temperatura Media (°C)',
            'Precipitación diaria (mm)'
        ),
        row_heights=[0.50, 0.25, 0.25]
    )

    # ── Demanda ──
    fig.add_trace(go.Scatter(
        x=d['fecha'], y=d['demanda_mw'],
        name='Demanda diaria',
        line=dict(color=COL_DEMANDA, width=1),
        opacity=0.45,
        hovertemplate='%{x|%d %b %Y}<br>Demanda: <b>%{y:,.0f} MW</b><extra></extra>'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=d['fecha'], y=d['media_movil'],
        name=f'Media móvil {ventana}d',
        line=dict(color='#0A3A6B', width=2.5),
        hovertemplate='Media móvil: <b>%{y:,.0f} MW</b><extra></extra>'
    ), row=1, col=1)

    if mostrar_festivos and len(festivos) > 0:
        fig.add_trace(go.Scatter(
            x=festivos['fecha'], y=festivos['demanda_mw'],
            mode='markers',
            name='Festivo nacional',
            marker=dict(color=COL_FESTIVO, size=6, symbol='circle',
                        line=dict(color='white', width=1)),
            hovertemplate='%{x|%d %b %Y} 🎉<br>Demanda: <b>%{y:,.0f} MW</b><extra></extra>'
        ), row=1, col=1)

    # ── Temperatura ──
    fig.add_trace(go.Scatter(
        x=d['fecha'], y=d['temp_max'],
        name='Tª Máxima', line=dict(color='#E07B39', width=1, dash='dot'),
        opacity=0.6,
        hovertemplate='Tª máx: <b>%{y:.1f}°C</b><extra></extra>'
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=d['fecha'], y=d['temp_med'],
        name='Tª Media', line=dict(color=COL_TEMP, width=2),
        hovertemplate='Tª media: <b>%{y:.1f}°C</b><extra></extra>'
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=d['fecha'], y=d['temp_min'],
        name='Tª Mínima', line=dict(color='#5B8FBF', width=1, dash='dot'),
        opacity=0.6,
        hovertemplate='Tª mín: <b>%{y:.1f}°C</b><extra></extra>'
    ), row=2, col=1)

    # ── Precipitación ──
    fig.add_trace(go.Bar(
        x=d['fecha'], y=d['precipitacion'],
        name='Precipitación',
        marker_color=COL_LLUVIA,
        hovertemplate='Lluvia: <b>%{y:.1f} mm</b><extra></extra>'
    ), row=3, col=1)

    # Layout
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=titulo('⚡ Serie Temporal de Demanda Eléctrica & Clima',
                        f'España · {anio_ini}–{anio_fin} · {len(d):,} observaciones diarias'),
            x=0.02, xanchor='left'
        ),
        height=750,
    )
    for axis in ['xaxis', 'xaxis2', 'xaxis3', 'yaxis', 'yaxis2', 'yaxis3']:
        fig.update_layout(**{axis: AXIS_STYLE})
    fig.update_yaxes(title_text='MW', row=1, col=1)
    fig.update_yaxes(title_text='°C', row=2, col=1)
    fig.update_yaxes(title_text='mm', row=3, col=1)

    with out1:
        out1.clear_output(wait=True)
        fig.show()

# Conectar widgets
controles1 = widgets.interactive(
    render_panel1,
    anio_ini=w_anio_inicio,
    anio_fin=w_anio_fin,
    ventana=w_media_movil,
    mostrar_festivos=w_mostrar_festivos
)

fila_filtros = widgets.HBox(
    [w_anio_inicio, w_anio_fin, w_media_movil, w_mostrar_festivos],
    layout=widgets.Layout(gap='20px', align_items='center', padding='10px 0')
)

display(HTML('<h3 style="font-family:sans-serif;color:#1A1A2E;margin-bottom:4px">🔎 Filtros — Serie Temporal</h3>'))
display(fila_filtros)
display(out1)

# Renderizado inicial
render_panel1(AÑOS[0], AÑOS[-1], 30, True)

---
## 📌 Panel 2 · KPIs y Comparativa Festivos vs Laborables

In [ ]:
# ── CELDA 5: Panel 2 – KPIs + Festivos vs Laborables ─────────────────────────
w2_anio = widgets.SelectMultiple(
    options=AÑOS,
    value=AÑOS[-3:],
    description='Años:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='180px', height='120px')
)
out2 = widgets.Output()

DIAS_ES = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
MESES_ES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

def render_panel2(años_sel):
    if not años_sel:
        with out2:
            out2.clear_output()
            print('⚠️  Selecciona al menos un año.')
        return

    d = df[df['anio'].isin(años_sel)].copy()

    # ── KPIs ──
    dem_media   = d['demanda_mw'].mean()
    dem_max     = d['demanda_mw'].max()
    dem_min     = d['demanda_mw'].min()
    pct_festivo = d['es_festivo'].mean() * 100
    media_fest  = d[d['es_festivo']==1]['demanda_mw'].mean()
    media_lab   = d[d['es_festivo']==0]['demanda_mw'].mean()
    caida_pct   = (media_lab - media_fest) / media_lab * 100

    kpi_html = f"""
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin:12px 0;font-family:'Helvetica Neue',sans-serif">
      <div style="background:#1B6CA8;color:white;padding:16px 22px;border-radius:10px;min-width:150px">
        <div style="font-size:11px;opacity:.8;text-transform:uppercase;letter-spacing:.05em">Demanda Media</div>
        <div style="font-size:26px;font-weight:700;margin-top:4px">{dem_media:,.0f}</div>
        <div style="font-size:11px;opacity:.7">MW / día</div>
      </div>
      <div style="background:#0A3A6B;color:white;padding:16px 22px;border-radius:10px;min-width:150px">
        <div style="font-size:11px;opacity:.8;text-transform:uppercase;letter-spacing:.05em">Pico Máximo</div>
        <div style="font-size:26px;font-weight:700;margin-top:4px">{dem_max:,.0f}</div>
        <div style="font-size:11px;opacity:.7">MW registrado</div>
      </div>
      <div style="background:#2A9D8F;color:white;padding:16px 22px;border-radius:10px;min-width:150px">
        <div style="font-size:11px;opacity:.8;text-transform:uppercase;letter-spacing:.05em">Media Laborable</div>
        <div style="font-size:26px;font-weight:700;margin-top:4px">{media_lab:,.0f}</div>
        <div style="font-size:11px;opacity:.7">MW / día</div>
      </div>
      <div style="background:#E63946;color:white;padding:16px 22px;border-radius:10px;min-width:150px">
        <div style="font-size:11px;opacity:.8;text-transform:uppercase;letter-spacing:.05em">Media Festivo</div>
        <div style="font-size:26px;font-weight:700;margin-top:4px">{media_fest:,.0f}</div>
        <div style="font-size:11px;opacity:.7">MW / día</div>
      </div>
      <div style="background:#F4A261;color:white;padding:16px 22px;border-radius:10px;min-width:150px">
        <div style="font-size:11px;opacity:.8;text-transform:uppercase;letter-spacing:.05em">Caída en Festivos</div>
        <div style="font-size:26px;font-weight:700;margin-top:4px">−{caida_pct:.1f}%</div>
        <div style="font-size:11px;opacity:.7">vs días laborables</div>
      </div>
    </div>
    """
    display(HTML(kpi_html))

    # ── Gráfica 1: Box por tipo de día ──
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            'Distribución por tipo de día',
            'Demanda media por día de semana',
            'Demanda media por mes'
        ),
        horizontal_spacing=0.10
    )

    colores_tipo = {'Laboral': COL_LABORAL, 'Festivo': COL_FESTIVO}
    for tipo, color in colores_tipo.items():
        sub = d[d['tipo_dia'] == tipo]
        fig.add_trace(go.Box(
            y=sub['demanda_mw'], name=tipo,
            marker_color=color,
            boxmean=True,
            hovertemplate=f'<b>{tipo}</b><br>%{{y:,.0f}} MW<extra></extra>'
        ), row=1, col=1)

    # ── Gráfica 2: Media por día de semana ──
    media_dia = d.groupby('dia_semana')['demanda_mw'].mean().reset_index()
    media_dia['nombre'] = media_dia['dia_semana'].map(lambda x: DIAS_ES[x])
    colores_bar = [COL_FESTIVO if d in [5,6] else COL_DEMANDA for d in media_dia['dia_semana']]
    fig.add_trace(go.Bar(
        x=media_dia['nombre'], y=media_dia['demanda_mw'],
        name='Media por día',
        marker_color=colores_bar,
        hovertemplate='<b>%{x}</b><br>Media: <b>%{y:,.0f} MW</b><extra></extra>',
        showlegend=False
    ), row=1, col=2)

    # ── Gráfica 3: Media por mes ──
    media_mes = d.groupby('mes')['demanda_mw'].mean().reset_index()
    media_mes['nombre'] = media_mes['mes'].map(lambda x: MESES_ES[x-1])
    fig.add_trace(go.Bar(
        x=media_mes['nombre'], y=media_mes['demanda_mw'],
        name='Media por mes',
        marker=dict(
            color=media_mes['demanda_mw'],
            colorscale='Blues',
            showscale=False
        ),
        hovertemplate='<b>%{x}</b><br>Media: <b>%{y:,.0f} MW</b><extra></extra>',
        showlegend=False
    ), row=1, col=3)

    años_str = ', '.join(map(str, sorted(años_sel)))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=titulo('📅 Comparativa Festivos vs Laborables', f'Años seleccionados: {años_str}'),
            x=0.02, xanchor='left'
        ),
        height=440,
        showlegend=True,
    )
    for axis in ['xaxis', 'xaxis2', 'xaxis3', 'yaxis', 'yaxis2', 'yaxis3']:
        fig.update_layout(**{axis: AXIS_STYLE})
    fig.update_yaxes(title_text='MW')

    with out2:
        out2.clear_output(wait=True)
        fig.show()

w2_obs = widgets.interactive_output(render_panel2, {'años_sel': w2_anio})

display(HTML('<h3 style="font-family:sans-serif;color:#1A1A2E;margin-bottom:4px">🔎 Filtro — Selecciona años (Ctrl+clic para varios)</h3>'))
display(w2_anio)
display(out2)

render_panel2(tuple(AÑOS[-3:]))

---
## 📌 Panel 3 · Correlación Temperatura ↔ Demanda

In [ ]:
# ── CELDA 6: Panel 3 – Correlación temperatura-demanda ───────────────────────
w3_anio_ini = widgets.Dropdown(
    options=AÑOS, value=AÑOS[0], description='Desde:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='160px')
)
w3_anio_fin = widgets.Dropdown(
    options=AÑOS, value=AÑOS[-1], description='Hasta:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='160px')
)
w3_tipo = widgets.ToggleButtons(
    options=['Todos', 'Solo Laborables', 'Solo Festivos'],
    value='Todos',
    description='Días:',
    style={'description_width': '50px', 'button_width': '130px'}
)
out3 = widgets.Output()

def render_panel3(anio_ini, anio_fin, tipo_filtro):
    d = df[(df['anio'] >= anio_ini) & (df['anio'] <= anio_fin)].copy()
    if tipo_filtro == 'Solo Laborables':
        d = d[d['es_festivo'] == 0]
    elif tipo_filtro == 'Solo Festivos':
        d = d[d['es_festivo'] == 1]

    d = d.dropna(subset=['temp_med', 'demanda_mw'])
    corr = d['temp_med'].corr(d['demanda_mw'])

    # Línea de tendencia (regresión lineal simple)
    coef = np.polyfit(d['temp_med'], d['demanda_mw'], 1)
    x_line = np.linspace(d['temp_med'].min(), d['temp_med'].max(), 200)
    y_line = np.polyval(coef, x_line)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Scatter: Temperatura vs Demanda  (r = {corr:.3f})',
            'Demanda media por tramo de temperatura'
        ),
        column_widths=[0.60, 0.40],
        horizontal_spacing=0.10
    )

    # ── Scatter con coloring por mes ──
    fig.add_trace(go.Scatter(
        x=d['temp_med'], y=d['demanda_mw'],
        mode='markers',
        name='Observación diaria',
        marker=dict(
            color=d['mes'],
            colorscale='RdYlBu_r',
            size=5,
            opacity=0.55,
            colorbar=dict(
                title='Mes', thickness=12,
                tickvals=list(range(1,13)),
                ticktext=MESES_ES,
                len=0.75
            ),
            showscale=True
        ),
        hovertemplate='Tª: <b>%{x:.1f}°C</b>  Demanda: <b>%{y:,.0f} MW</b><extra></extra>'
    ), row=1, col=1)

    # Línea de tendencia
    fig.add_trace(go.Scatter(
        x=x_line, y=y_line,
        mode='lines',
        name='Tendencia lineal',
        line=dict(color=COL_FESTIVO, width=2.5, dash='dash'),
        hovertemplate='Tendencia: <b>%{y:,.0f} MW</b><extra></extra>'
    ), row=1, col=1)

    # ── Barras por tramo de temperatura ──
    bins = pd.cut(d['temp_med'], bins=10)
    media_tramo = d.groupby(bins, observed=True)['demanda_mw'].mean().reset_index()
    media_tramo['etiqueta'] = media_tramo['temp_med'].astype(str).str.replace(r'[\(\]\s]', '', regex=True)
    fig.add_trace(go.Bar(
        x=media_tramo['etiqueta'],
        y=media_tramo['demanda_mw'],
        name='Media por tramo Tª',
        marker=dict(
            color=media_tramo['demanda_mw'],
            colorscale='RdYlBu_r',
            showscale=False
        ),
        hovertemplate='Tramo: <b>%{x}</b><br>Media: <b>%{y:,.0f} MW</b><extra></extra>',
        showlegend=False
    ), row=1, col=2)

    años_str = f'{anio_ini}–{anio_fin}'
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=titulo('🌡️ Correlación Temperatura ↔ Demanda Eléctrica',
                        f'{años_str} · {tipo_filtro} · {len(d):,} observaciones · Pearson r = {corr:.3f}'),
            x=0.02, xanchor='left'
        ),
        height=500,
    )
    for axis in ['xaxis', 'xaxis2', 'yaxis', 'yaxis2']:
        fig.update_layout(**{axis: AXIS_STYLE})
    fig.update_xaxes(title_text='Temperatura media (°C)', row=1, col=1)
    fig.update_xaxes(title_text='Tramo de Tª (°C)', row=1, col=2, tickangle=-45)
    fig.update_yaxes(title_text='Demanda (MW)')

    with out3:
        out3.clear_output(wait=True)
        fig.show()

w3_obs = widgets.interactive_output(
    render_panel3,
    {'anio_ini': w3_anio_ini, 'anio_fin': w3_anio_fin, 'tipo_filtro': w3_tipo}
)

fila3 = widgets.HBox(
    [w3_anio_ini, w3_anio_fin, w3_tipo],
    layout=widgets.Layout(gap='20px', align_items='center', padding='10px 0')
)

display(HTML('<h3 style="font-family:sans-serif;color:#1A1A2E;margin-bottom:4px">🔎 Filtros — Correlación</h3>'))
display(fila3)
display(out3)

render_panel3(AÑOS[0], AÑOS[-1], 'Todos')

---
## 📌 Panel 4 · Mapa de Calor Mensual por Año

In [ ]:
# ── CELDA 7: Panel 4 – Heatmap año × mes ─────────────────────────────────────
w4_metrica = widgets.RadioButtons(
    options=['demanda_mw', 'temp_med', 'precipitacion'],
    value='demanda_mw',
    description='Métrica:',
    style={'description_width': '70px'}
)
out4 = widgets.Output()

LABELS = {
    'demanda_mw':    ('Demanda media mensual (MW)',   'Blues'),
    'temp_med':      ('Temperatura media mensual (°C)', 'RdYlBu_r'),
    'precipitacion': ('Precipitación media mensual (mm)', 'YlGnBu'),
}

def render_panel4(metrica):
    tabla = df.groupby(['anio', 'mes'])[metrica].mean().reset_index()
    pivot = tabla.pivot(index='anio', columns='mes', values=metrica)
    pivot.columns = MESES_ES

    label, escala = LABELS[metrica]

    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=MESES_ES,
        y=pivot.index.tolist(),
        colorscale=escala,
        hovertemplate='<b>%{y} · %{x}</b><br>Valor: <b>%{z:,.1f}</b><extra></extra>',
        colorbar=dict(title=metrica, thickness=14),
        xgap=2, ygap=2
    ))

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=titulo(f'🗓️ Mapa de Calor — {label}', 'Media mensual por año'),
            x=0.02, xanchor='left'
        ),
        height=420,
        xaxis=dict(**AXIS_STYLE, title='Mes'),
        yaxis=dict(**AXIS_STYLE, title='Año', dtick=1),
    )

    with out4:
        out4.clear_output(wait=True)
        fig.show()

w4_obs = widgets.interactive_output(render_panel4, {'metrica': w4_metrica})

display(HTML('<h3 style="font-family:sans-serif;color:#1A1A2E;margin-bottom:4px">🔎 Selecciona la métrica</h3>'))
display(w4_metrica)
display(out4)

render_panel4('demanda_mw')

---
## 📌 Panel 5 · Evolución Interanual Comparada

In [ ]:
# ── CELDA 8: Panel 5 – Evolución interanual superpuesta ──────────────────────
w5_años = widgets.SelectMultiple(
    options=AÑOS,
    value=AÑOS[-4:],
    description='Años:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='180px', height='130px')
)
out5 = widgets.Output()

# Paleta diferenciada para superponer años
PALETA = px.colors.qualitative.Plotly

def render_panel5(años_sel):
    if not años_sel:
        return
    d = df[df['anio'].isin(años_sel)].copy()
    d['dia_año'] = d['fecha'].dt.dayofyear

    fig = go.Figure()
    for i, anio in enumerate(sorted(años_sel)):
        sub = d[d['anio'] == anio].sort_values('dia_año')
        sub['smooth'] = sub['demanda_mw'].rolling(14, center=True).mean()
        color = PALETA[i % len(PALETA)]
        fig.add_trace(go.Scatter(
            x=sub['dia_año'],
            y=sub['smooth'],
            name=str(anio),
            line=dict(color=color, width=2.2),
            hovertemplate=f'<b>{anio}</b> · Día %{{x}}<br>Demanda: <b>%{{y:,.0f}} MW</b><extra></extra>'
        ))

    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=titulo('📈 Evolución Interanual Comparada',
                        'Demanda diaria suavizada (media móvil 14 días) · eje X = día del año'),
            x=0.02, xanchor='left'
        ),
        height=460,
        xaxis=dict(**AXIS_STYLE, title='Día del año',
                   tickvals=[1,32,60,91,121,152,182,213,244,274,305,335],
                   ticktext=MESES_ES),
        yaxis=dict(**AXIS_STYLE, title='Demanda (MW)'),
    )

    with out5:
        out5.clear_output(wait=True)
        fig.show()

w5_obs = widgets.interactive_output(render_panel5, {'años_sel': w5_años})

display(HTML('<h3 style="font-family:sans-serif;color:#1A1A2E;margin-bottom:4px">🔎 Selecciona años a comparar (Ctrl+clic)</h3>'))
display(w5_años)
display(out5)

render_panel5(tuple(AÑOS[-4:]))